# 📓 Notebook 04 — Databases

SQLi beyond sqlmap defaults, NoSQL injection, unauthenticated Redis/Mongo, MSSQL xp_cmdshell, credential cracking.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

Wed Oct 14 07:47:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

GPU: Tesla T4 | VRAM 15.6 GB | RAM 13.6 GB


overlay         236G   81G  156G  35% /


In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


cache: /content/drive/MyDrive/redteam


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

Server Version: 29.1.3-colab
Storage Driver: vfs
docker ready


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

/usr/local/bin/ollama
{"version":"0.40.4"} <- ollama up


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


qwen2.5-coder:7b ready
NAME                ID              SIZE      MODIFIED       
gpt-oss:20b         f38aa0c53da5    13 GB     26 minutes ago    
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    48 minutes ago    


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


hexstrike:
{"all_essential_tools_available":false,"cache_stats":{"evictions":0,"hit_rate":"10.2%","hits":258,"max_size":1000,"misses":2282,"size":20},"category_stats":{"additional":{"available":1,"total":14},"api":{"available":1,"total":8},"binary":{"available":3,"total":13},"cloud":{"available":0,"total":10},"essential":{"available":4,"total":8},"exploitation":{"available":0,"total":3},"forensics":{"available":2,"total":16},"network":{"available":3,"total":10},"osint":{"available":1,"total":13},"password":{"available":0,"total":5},"vuln_scanning":{"available":1,"total":4},"web_security":{"available":4,"total":19},"wireless":{"available":0,"total":4}},"message":"HexStrike AI Tools API Server is operational","status":"healthy","telemetry":{"average_execution_time":"0.00s","commands_executed":2282,"success_rate":"0.9%","system_metrics":{"cpu_percent":5.6,"disk_usage":34.1,"memory_percent":28.9,"network_io":{"bytes_recv":37115416614,"bytes_sent":145264306,"dropin":0,"dropout":0,"errin":0,

In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


HexStrike URL:
https://modification-similarly-retention-theatre.trycloudflare.com


In [ ]:
%%bash
apt-get -qq update && apt-get -qq install -y default-mysql-client postgresql-client redis-tools || true
pip -q install pymongo pymysql psycopg2-binary redis sqlmap 2>/dev/null || true
[ ! -d /content/nosqlmap ] && git clone --depth 1 https://github.com/codingo/NoSQLMap /content/nosqlmap 2>/dev/null || true
[ ! -d /content/odat ] && git clone --depth 1 https://github.com/quentinhardy/odat /content/odat 2>/dev/null || true
echo "db tools ready"


Selecting previously unselected package mysql-client-core-8.0.
(Reading database ... 172391 files and directories currently installed.)
Preparing to unpack .../0-mysql-client-core-8.0_8.0.46-0ubuntu0.24.04.4_amd64.deb ...
Unpacking mysql-client-core-8.0 (8.0.46-0ubuntu0.24.04.4) ...
Selecting previously unselected package mysql-client-8.0.
Preparing to unpack .../1-mysql-client-8.0_8.0.46-0ubuntu0.24.04.4_amd64.deb ...
Unpacking mysql-client-8.0 (8.0.46-0ubuntu0.24.04.4) ...
Selecting previously unselected package default-mysql-client.
Preparing to unpack .../2-default-mysql-client_1.1.0build1_all.deb ...
Unpacking default-mysql-client (1.1.0build1) ...
Selecting previously unselected package libjemalloc2:amd64.
Preparing to unpack .../3-libjemalloc2_5.3.0-2build1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.3.0-2build1) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../4-liblzf1_3.6-4_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-4) ...
Selecting previou

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
# --- vulnerable-by-config targets ---
docker run -d --name mysql --network host -e MYSQL_ROOT_PASSWORD=root -e MYSQL_DATABASE=vuln mysql:8
docker run -d --name pgsql --network host -e POSTGRES_PASSWORD=postgres postgres:16
docker run -d --name mongo --network host mongo:7
docker run -d --name redis --network host redis:7
# MSSQL (SA / weak pw)
docker run -d --name mssql --network host -e ACCEPT_EULA=Y -e MSSQL_SA_PASSWORD='Password123!' mcr.microsoft.com/mssql/server:2022-latest
sleep 45
nmap -sV localhost -p 3306,5432,27017,6379,1433 | tail -20

Container mysql started
Container pgsql started
Container mongo started
Container redis started
Container mssql started
27017/tcp open   mongod?
1 service unrecognized despite returning data. If you know the service/version, please submit the following fingerprint at https://nmap.org/cgi-bin/submit.cgi?new-service :
SF-Port27017-TCP:V=7.94SVN%I=7%D=10/14%Time=6ACF3494%P=x86_64-pc-linux-gnu
SF:%r(mongodb,114,"\x14\x01\0\0\x03\0\0\0:0\0\0\x01\0\0\0\x08\0\0\0\0\0\0\
SF:0\0\0\0\0\0\0\0\0\x01\0\0\0\xf0\0\0\0\x01ok\0\0\0\0\0\0\0\0\0\x02errmsg
SF:\0\xa1\0\0\0Unsupported\x20OP_QUERY\x20command:\x20serverStatus\.\x20Th
SF:e\x20client\x20driver\x20may\x20require\x20an\x20upgrade\.\x20For\x20mo
SF:re\x20details\x20see\x20https://dochub\.mongodb\.org/core/legacy-opcode
SF:-removal\0\x10code\0`\x01\0\0\x02codeName\0\x1a\0\0\0UnsupportedOpQuery
SF:Command\0\0")%r(GetRequest,A9,"HTTP/1\.0\x20200\x20OK\r\nConnection:\x2
SF:0close\r\nContent-Type:\x20text/plain\r\nContent-Length:\x2085\r\n\r\nI
SF:t\x2

In [ ]:
# --- example attacks ---
!redis-cli -h localhost INFO server 2>/dev/null | head || true
!redis-cli -h localhost CONFIG GET dir 2>/dev/null || true
!mysql -h 127.0.0.1 -uroot -proot -e "SELECT version();" vuln 2>/dev/null || true
!sqlmap -u "http://localhost:3000/rest/products/search?q=1" --batch --level=1 --dbs 2>/dev/null | head -30 || true


        ___
       __H__
 ___ ___[.]_____ ___ ___  {1.10.10#pip}
|_ -| . [(]     | .'| . |
|___|_  [(]_|_|_|__,|  _|
      |_|V...       |_|   https://sqlmap.org

[!] legal disclaimer: Usage of sqlmap for attacking targets without prior mutual consent is illegal. It is the end user's responsibility to obey all applicable local, state and federal laws. Developers assume no liability and are not responsible for any misuse or damage caused by this program

[*] starting @ 07:52:32 /2026-10-14/

[07:52:32] [INFO] testing connection to the target URL
[07:52:33] [CRITICAL] unable to connect to the target URL ('Connection refused'). sqlmap is going to retry the request(s)
[07:52:33] [WARNING] if the problem persists please check that the provided target URL is reachable. In case that it is, you can try to rerun with switch '--random-agent' and/or proxy switches ('--proxy', '--proxy-file'...)


[07:52:33] [CRITICAL] unable to connect to the target URL ('Connection refused')

[*] ending @ 07:52:33 /2026-10-14/

